In [1]:
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
import os
import string
from collections import defaultdict
import copy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
docdf = pd.read_csv('IRdata_Resource.csv')

In [3]:
docdf

,Unnamed: 0,domain,subdomain,text,link
0,0,Structure and Bonding,Introduction to Organic Chemistry,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
1,1,Structure and Bonding,Atomic Structure - The Nucleus,"Objective After completing this section, you ...",https://chem.libretexts.org/Bookshelves/Organi...
2,2,Structure and Bonding,Atomic Structure - Orbitals,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
3,3,Structure and Bonding,Atomic Structure - Electron Configurations,"Objective After completing this section, you ...",https://chem.libretexts.org/Bookshelves/Organi...
4,4,Structure and Bonding,Development of Chemical Bonding Theory,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
...,...,...,...,...,...
351,351,Synthetic Polymers,Stereochemistry of Polymerization: Ziegler-Nat...,The following diagram presents one mechanism ...,https://chem.libretexts.org/Bookshelves/Organi...
352,352,Synthetic Polymers,Copolymers,Graft polymers can be made in great profusion...,https://chem.libretexts.org/Bookshelves/Organi...
353,353,Synthetic Polymers,Step-Growth Polymers,The process of step-growth polymerizations ar...,https://chem.libretexts.org/Bookshelves/Organi...
354,354,Synthetic Polymers,Olefin Metathesis Polymerization,The product polymer of a ROMP using cyclopen...,https://chem.libretexts.org/Bookshelves/Organi...


In [4]:
docdf.drop([docdf.columns[0]], axis=1, inplace=True)

In [5]:
docdf.head()

,domain,subdomain,text,link
0,Structure and Bonding,Introduction to Organic Chemistry,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
1,Structure and Bonding,Atomic Structure - The Nucleus,"Objective After completing this section, you ...",https://chem.libretexts.org/Bookshelves/Organi...
2,Structure and Bonding,Atomic Structure - Orbitals,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
3,Structure and Bonding,Atomic Structure - Electron Configurations,"Objective After completing this section, you ...",https://chem.libretexts.org/Bookshelves/Organi...
4,Structure and Bonding,Development of Chemical Bonding Theory,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...


In [6]:
docdf.shape

(356, 4)

In [7]:
docs = []
doclinks = []

for i in range(docdf.shape[0]):
    doclinks.append(docdf['link'][i])
    docs.append(docdf['text'][i])

## Preprocessing

In [8]:
lemmer = nltk.WordNetLemmatizer()

def lemmatizer(tokens):
    return [lemmer.lemmatize(token) for token in tokens]

In [9]:
def removePunctuation(tokens):
    return[t for t in tokens if t not in string.punctuation]

In [10]:
stop_words = set(stopwords.words('english'))

In [11]:
def preprocess(doc):
    procd = []
    line = doc.lower()
    line = word_tokenize(line)
    line = removePunctuation(line)
    line = lemmatizer(line)
    for word in line:
        if(word not in stop_words):
            procd.append(word)
    return procd

In [12]:
prodocs = []

for doc in docs:
    procd = preprocess(doc)
    prodocs.append(procd)

In [13]:
len(prodocs)

356

## TF-IDF

In [14]:
from collections import Counter

def defaultdnorm():
    return 0.5

def defaultidf():
    return np.log(len(prodocs))

In [15]:
vocab = set()
for doc in prodocs:
    for word in doc:
        if word not in vocab:
            vocab.add(word)

vocab = list(vocab)

In [16]:
len(vocab)

15078

In [17]:
class tfidf:
    def __init__(self):
        self.idf = defaultdict(defaultidf)
        self.tfeq = defaultdict(lambda: defaultdict(int))

    def computetf(self, prodocs):
        for i in range(len(prodocs)):
            doc = prodocs[i]
            countdata = Counter(doc)
            for word in doc:
                count = countdata[word]
                self.tfeq[i][word] = count / len(doc)

    def computeidf(self, prodocs):
        temp = defaultdict(set)
        num = len(prodocs)
        for i in range(len(prodocs)):
            doc = prodocs[i]
            for word in doc:
                temp[word].add(i)

        for word in vocab:
            self.idf[word] = np.log(num / (1 + len(temp[word])))

In [18]:
obj = tfidf()
obj.computetf(prodocs)
obj.computeidf(prodocs)

## TF-IDF matrices

In [19]:
tfidfmat = np.zeros((len(prodocs), len(vocab)))

vindex = dict()
for i in range(len(vocab)):
    vindex[vocab[i]] = i


for i in range(len(prodocs)):
    doc = prodocs[i]
    for word in doc:
        ind = vindex[word]
        idf = obj.idf[word]
        tfidfmat[i][ind] = obj.tfeq[i][word] * idf

In [20]:
# Query = question that user could not solve

df = pd.read_csv('IRdata2.csv')
query = df['question'][1184]
#query = input('Enter the query: ')
tokens = preprocess(query)
tokens

['one',
 'molecule',
 'dialkyl',
 'ether',
 'produce',
 'many',
 'molecule',
 'alkyl',
 'halide',
 'excess',
 'halogen',
 'acid']

In [21]:
query

'One molecule of dialkyl ether produces how many molecules of alkyl halides with excess of halogen acid?'

## Cosine similarity

In [22]:
def genqvec(tokens):
    qvec = np.zeros(len(vocab))

    qobj = tfidf()
    qobj.computetf([tokens])

    for word in tokens:
        if(word in vocab):
            ind = vindex[word]
            idf = obj.idf[word]
            qvec[ind] = qobj.tfeq[0][word] * idf
    
    return qvec

In [23]:
def cossim(d1, d2):
    num = np.dot(d1, d2)
    denom = np.linalg.norm(d1) * np.linalg.norm(d2)
    cossim = num / denom
    return cossim

In [24]:
qvec = genqvec(tokens)

In [25]:
cosscores = []

for i in range(len(prodocs)):
    cosscores.append(cossim(qvec, tfidfmat[i]))
    
out = np.array(cosscores).argsort()[-10:][::-1]


#print('The top 10 documents for each scheme using cosine similarity are:\n')
#print('Term Frequency:  ', out)

In [26]:
linklist = [doclinks[val] for val in out]
#print('Cosine similarity scores:\n')
#print('\n', linklist)
#print(sorted(cosscores, reverse=True)[:5])

In [27]:
out

array([212, 214, 222, 126, 211, 213, 218, 220, 118,  29], dtype=int64)

In [28]:
df.iloc[out]

,QID,domain,subdomain,question,imageURL,options,answer,explaination
212,212,Electrochemistry,Electrochemical Cells,Which of the following is a not a secondary cell?,0,"['Nickel-cadmium cell', 'Lead storage cell', '...",d,A secondary battery (a series of cells) is one...
214,214,Electrochemistry,Electrochemical Cells,What is the observation when the opposing exte...,0,['The electrochemical cell behaves like an ele...,a,"In an electrochemical cell, when an opposing e..."
222,222,Electrochemistry,Galvanic Cells,Which of the following is the correct order of...,0,"['Zn &gt; Mg &gt; Fe &gt; Cu &gt; Ag', 'Zn &gt...",d,"Greater the oxidation potential of metal, the ..."
126,126,Solutions,Solutions Concentration,What symbol is used to denote ‘molality’?,0,"['M', 'm', 'mM', 'n']",b,"Universally, ‘m’ is used to denote molality co..."
211,211,Electrochemistry,Electrochemical Cells,What is the direction of flow of electrons in ...,0,"['Anode to cathode externally', 'Anode to cath...",a,An electrolytic cell is a type of electrochemi...
213,213,Electrochemistry,Electrochemical Cells,Which of the following statements regarding pr...,0,"['Primary cells cannot be recharged', 'They ha...",b,Primary cells cannot be used again and again. ...
218,218,Electrochemistry,Galvanic Cells,Which of the following electrolytes is not pre...,0,"['KCl', 'KNO<sub>3</sub>', 'NH<sub>4</sub>NO<s...",d,"In a salt bridge, the electrolytes like KCl, K..."
220,220,Electrochemistry,Galvanic Cells,The electrode on which oxidation occurs is cal...,0,"['True', 'False']",a,An anode is an electrode where oxidation takes...
118,118,Solutions,Solutions Concentration,Iron (III) oxide chunks contain 80 ppm silica ...,0,"['0.008%', '0.080%', '0.800%', '8.000%']",a,80 ppm silica means there are 80 mg of silica ...
29,29,Solid State,Crystalline Solids,The molecules in polar molecular solid are hel...,0,"['dipole-dipole interaction', 'london forces',...",a,The force responsible for holding together the...


In [29]:
np.unique(df['domain'])

array(['Alcohols, Phenols & Ethers',
       'Aldehydes, Ketones & Carboxylic Acids', 'Amines', 'Biomolecules',
       'Chemical Kinetics', 'Coordination Compounds',
       'D & F-Block Elements', 'Electrochemistry',
       'Elements Principles & Processes', 'Everyday Life Chemistry',
       'Haloalkanes & Haloarenes', 'P-Block Elements', 'Polymers',
       'Solid State', 'Solutions', 'Surface Chemistry'], dtype=object)

In [30]:
df[df['domain'] == 'Alcohols, Phenols & Ethers']

,QID,domain,subdomain,question,imageURL,options,answer,explaination
1067,1067,"Alcohols, Phenols & Ethers",Alcohols Classification,What is the general formula for an aliphatic a...,0,"['R-H', 'R-OH', 'R-CHO', 'R-COOH']",b,Alcohols are compounds having OH group attache...
1068,1068,"Alcohols, Phenols & Ethers",Alcohols Classification,Which of the following is true regarding polyh...,0,"['It should have one or more OH groups', 'It s...",b,"Alcohols may be classified as mono-, di- tri- ..."
1069,1069,"Alcohols, Phenols & Ethers",Alcohols Classification,Which of the following types of alcohol contai...,0,"['Primary allylic alcohols', 'Secondary allyli...",d,Allylic alcohols are those in which OH group i...
1070,1070,"Alcohols, Phenols & Ethers",Alcohols Classification,Which of the following terms does not describe...,0,"['Primary', 'Monohydric', 'Allylic', 'Vinylic']",d,The given compound contains a sp<sup>3</sup> h...
1071,1071,"Alcohols, Phenols & Ethers",Alcohols Classification,Choose the most suitable classification for th...,0,"['<noscript><img alt=""The classification for c...",b,"The compound shown is a monohydric, tertiary a..."
...,...,...,...,...,...,...,...,...
1184,1184,"Alcohols, Phenols & Ethers",Ethers,One molecule of dialkyl ether produces how man...,0,"['1', '2', '3', '4']",b,Dialkyl ethers when treated with excess haloge...
1185,1185,"Alcohols, Phenols & Ethers",Ethers,What product(s) are formed when the shown ethe...,https://www.sanfoundry.com/wp-content/uploads/...,['(C<sub>6</sub>H<sub>5</sub>)CH<sub>2</sub>I ...,a,The C-O bond of the phenol group is stronger t...
1186,1186,"Alcohols, Phenols & Ethers",Ethers,What is the major product of bromination of an...,0,"['o-Dibromobenzene', 'p-Dibromobenzene', 'o-Br...",d,Phenyl alkyl ethers undergo bromination in the...
1187,1187,"Alcohols, Phenols & Ethers",Ethers,Identify the major product of Friedel-Crafts a...,0,"['2-Methoxytoluene', '4-Methoxytoluene', '2-Me...",d,Anisole undergoes Friedel-Crafts acylation whe...


In [31]:
class documentRetriever:
    def __init__(self):
        self.newdf = None
    
    def retrieve(self, query, k):
        tokens = preprocess(query)
        qvec = genqvec(tokens)
        
        cosscores = []
        for i in range(len(prodocs)):
            cosscores.append(cossim(qvec, tfidfmat[i]))

        out = np.array(cosscores).argsort()[-k:][::-1]
        self.newdf = docdf.iloc[out]

In [33]:
qdf = pd.read_csv('IRdata2.csv').drop(['imageURL'], axis=1)
qdf["QID"] = "CH" + qdf["QID"].map(str)

In [38]:
q1 = qdf['question'][1566]
q1

'Identify the correct formula for the carbohydrate rhamnose?'

In [39]:
dr = documentRetriever()
dr.retrieve(q1, 10)
dr.newdf

,domain,subdomain,text,link
297,Biomolecules- Carbohydrates,Introduction,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
331,Biomolecules - Lipids,Biomolecules - Lipids (Summary),Concepts & Vocabulary fulfill all of the deta...,https://chem.libretexts.org/Bookshelves/Organi...
12,Structure and Bonding,Drawing Chemical Structures,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
298,Biomolecules- Carbohydrates,Classification of Carbohydrates,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
309,Biomolecules- Carbohydrates,Biomolecules- Carbohydrates (Summary),Concepts & Vocabulary fulfill all of the deta...,https://chem.libretexts.org/Bookshelves/Organi...
308,Biomolecules- Carbohydrates,Cell-Surface Carbohydrates and Influenza Viruses,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
30,Organic Compounds- Alkanes and Their Stereoche...,Alkanes and Alkane Isomers,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
80,Alkenes- Structure and Reactivity,Calculating Degree of Unsaturation,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
323,Biomolecules - Lipids,Introduction to Lipids,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
13,Structure and Bonding,Structure and Bonding (Summary),Concepts & Vocabulary 1.0: Prelude to Structu...,https://chem.libretexts.org/Bookshelves/Organi...
